## General Bands — Multi-Participant

**Instructions:**
- In the *File Loading* cell, add one `(eeg_path, beh_path)` tuple per participant.
- `N_PARTICIPANTS` is counted automatically.
- Column name notes:
  - `"Stimulus"` — event-type column in the PsychoPy CSV
  - `"Marker Timestamp"` — event-time column in the PsychoPy CSV
  - Change those in `sync_and_mark` / `load_behav` if your files use different names.
- To adjust the artifact-rejection threshold, change the value in `self._clean_windows(threshold=50)` inside `Recording.__init__`.
- Use only forward slashes in file paths. No trailing spaces.

In [ ]:
!pip install mne PyWavelets

In [ ]:
import numpy as np
import pandas as pd
import pywt
import mne
from scipy import signal, integrate
import matplotlib.pyplot as plt

## File Loading

Add one `(eeg_path, beh_path)` tuple per participant.
Copy-paste and edit the lines — `N_PARTICIPANTS` is counted automatically.

In [ ]:
# ── ADD / REMOVE PARTICIPANTS HERE ────────────────────────────────────────────
# Each entry is (eeg_filepath, psychopy_filepath)

participant_files = [
    ("/path/to/participant_01_eeg.csv",  "/path/to/participant_01_beh.csv"),   # participant 1
    ("/path/to/participant_02_eeg.csv",  "/path/to/participant_02_beh.csv"),   # participant 2
    # ("/path/to/participant_03_eeg.csv",  "/path/to/participant_03_beh.csv"),  # participant 3
    # ... keep adding lines, one per participant
]

# ── AUTO-COUNTED — do not edit ─────────────────────────────────────────────────
N_PARTICIPANTS = len(participant_files)
print(f"Loaded file paths for {N_PARTICIPANTS} participant(s).")

## Helper Functions

In [ ]:
def load_eeg(eeg_csv: str) -> pd.DataFrame:
    df = pd.read_csv(eeg_csv, sep=None, engine='python')
    df['timestamps'] = pd.to_numeric(df['timestamps'], errors='coerce')
    df = df[df['timestamps'].notnull()]
    return df


def load_behav(behav_csv: str) -> pd.DataFrame:
    """
    Reads the PsychoPy behavioural CSV and aligns timestamps to Unix seconds.
    Change 'Marker Timestamp' and 'Stimulus' below if your column names differ.
    """
    beh = pd.read_csv(behav_csv)
    beh['Marker Timestamp'] = pd.to_numeric(beh['Marker Timestamp'], errors='coerce')
    beh = beh[beh['Marker Timestamp'].notnull()]
    beh['marker_ts'] = beh['Marker Timestamp']   # already Unix seconds
    return beh


def sync_and_mark(
    eeg_df: pd.DataFrame,
    beh_df: pd.DataFrame,
    marker_map: dict,
    fs: int = 256
) -> pd.DataFrame:
    start_ts = max(eeg_df['timestamps'].min(), beh_df['marker_ts'].min())
    end_ts   = min(eeg_df['timestamps'].max(), beh_df['marker_ts'].max())
    eeg_sync = eeg_df.query("@start_ts <= timestamps <= @end_ts").copy()
    eeg_sync['marker'] = 0
    times = eeg_sync['timestamps'].values
    for _, row in beh_df.iterrows():
        code = marker_map.get(row['Stimulus'], 0)
        idx  = np.argmin(np.abs(times - row['marker_ts']))
        eeg_sync.at[eeg_sync.index[idx], 'marker'] = code
    return eeg_sync.reset_index(drop=True)

## Recording & Grand Average Classes

In [ ]:
def apply_ica(raw, n_components, eog_thresh=2.5):
    data = raw.get_data()
    if data.shape[1] < 20 or np.isnan(data).any() or np.allclose(data, 0):
        print("⚠️ Skipping ICA: insufficient data duration or invalid signal")
        return raw
    ica = mne.preprocessing.ICA(
        n_components=n_components,
        max_iter='auto',
        random_state=0
    )
    try:
        ica.fit(raw)
    except Exception as e:
        print(f"⚠️ ICA failed: {e}")
        return raw
    try:
        eog_indices, scores = ica.find_bads_eog(raw, threshold=eog_thresh)
        ica.exclude = eog_indices
        raw = ica.apply(raw.copy())
    except Exception as e:
        print(f"⚠️ EOG removal skipped: {e}")
    return raw


def wavelet_denoise(sig, wavelet='db4', level=4):
    coeffs = pywt.wavedec(sig, wavelet, level=level)
    sigma  = np.median(np.abs(coeffs[-level])) / 0.6745
    uthresh = sigma * np.sqrt(2 * np.log(len(sig)))
    coeffs[1:] = [pywt.threshold(c, uthresh, mode='hard') for c in coeffs[1:]]
    return pywt.waverec(coeffs, wavelet)[:len(sig)]


class Recording:
    def __init__(
        self,
        eeg_df: pd.DataFrame,
        beh_df: pd.DataFrame,
        marker_map: dict,
        length: float = 20,
        bands: dict = None,
        fs: int = 256,
        min_freq: float = 1,
        max_freq: float = 30
    ):
        self.fs = fs
        self.length = length
        self.marker_map = marker_map
        self.marker_dict = {v: k for k, v in marker_map.items()}
        self.bands = bands or {
            "delta": [0.5, 4], "theta": [4, 8],
            "alpha": [8, 12],  "beta":  [12, 30]
        }
        self.beh = beh_df

        merged = sync_and_mark(eeg_df, beh_df, marker_map, fs)
        picks  = ['TP9', 'AF7', 'AF8', 'TP10']

        data = merged[picks].T.values * 1e-6
        info = mne.create_info(picks, fs, ['eeg'] * 4)
        raw  = mne.io.RawArray(data, info, verbose=False)
        raw.filter(min_freq, max_freq, method='iir', verbose=False)
        n_ica_components = min(len(picks), raw.info['nchan'], raw.get_data().shape[0])
        raw = apply_ica(raw, n_components=n_ica_components)

        clean = raw.get_data().T * 1e6
        for i, ch in enumerate(picks):
            merged[ch] = clean[:, i]

        self.df = merged
        self._epoch_fixed_length(picks)
        self._clean_windows(threshold=50)  # CHANGE THRESHOLD AS NEEDED
        self._compute_welch()
        self._compute_band_power()

    def _epoch_fixed_length(self, picks):
        n_pts = int(self.length * self.fs)
        self.events = {code: {ch: [] for ch in picks} for code in self.marker_dict}
        idxs = self.df.index[self.df['marker'] != 0]
        for idx in idxs:
            code = int(self.df.at[idx, 'marker'])
            for ch in picks:
                segment = self.df[ch].iloc[idx: idx + n_pts].values
                if len(segment) == n_pts:
                    self.events[code][ch].append(segment)

    def _clean_windows(self, threshold=100):
        self.windows = {c: {ch: [] for ch in self.events[c]} for c in self.events}
        size = int(self.length * self.fs // 4)
        for c in self.events:
            for ch, segs in self.events[c].items():
                for seg in segs:
                    for j in range(0, len(seg), size):
                        w = seg[j: j + size]
                        if np.max(np.abs(w)) < threshold:
                            self.windows[c][ch].append(w)

    def _compute_welch(self):
        self.freqs = None
        self.psd = {c: {ch: None for ch in self.windows[c]} for c in self.windows}
        win = int(self.length * self.fs)
        for c in self.windows:
            for ch, ws in self.windows[c].items():
                psds, freqs_list = [], []
                for w in ws:
                    f, Pxx = signal.welch(w, fs=self.fs, nperseg=win)
                    psds.append(Pxx)
                    freqs_list.append(np.round(f, 3))
                if not psds:
                    continue
                freq_keys = [tuple(f) for f in freqs_list]
                most_common_freq_key = pd.Series(freq_keys).value_counts().idxmax()
                most_common_freq = np.array(most_common_freq_key)
                self.freqs = most_common_freq
                matched_psds = [ps for ps, f in zip(psds, freqs_list)
                                if np.allclose(f, most_common_freq)]
                self.psd[c][ch] = np.mean(matched_psds, axis=0) if matched_psds else None

    def _compute_band_power(self):
        if self.freqs is None:
            print("⚠️ No frequency data — skipping band power computation.")
            self.relative_power = {}
            self.band_df = pd.DataFrame()
            return
        res = {}
        freq_res = self.freqs[1] - self.freqs[0]
        for c in self.psd:
            res[c] = {}
            for ch, P in self.psd[c].items():
                if not isinstance(P, np.ndarray) or P.size == 0:
                    print(f"⚠️ Skipping '{ch}' in {self.marker_dict[c]} — invalid PSD")
                    continue
                try:
                    total = integrate.simpson(P, dx=freq_res)
                    res[c][ch] = {}
                    for b, (f0, f1) in self.bands.items():
                        idx = (self.freqs >= f0) & (self.freqs <= f1)
                        bp = integrate.simpson(P[idx], dx=freq_res)
                        res[c][ch][b] = bp / total
                except Exception as e:
                    print(f"⚠️ Band power error for '{ch}' in {self.marker_dict[c]}: {e}")
        self.relative_power = res
        records = []
        for c in res:
            label = self.marker_dict[c]
            for ch in res[c]:
                row = {'Condition': label, 'Channel': ch}
                row.update(res[c][ch])
                records.append(row)
        self.band_df = pd.DataFrame(records)

    def plot(self, condition: str, ylim=(0, 150), xlim=(0, 40)):
        inv  = {v: k for k, v in self.marker_dict.items()}
        code = inv.get(condition)
        plt.figure(dpi=120)
        for ch, P in self.psd[code].items():
            if P is None:
                print(f"⚠️ No PSD for '{ch}' — skipping")
                continue
            plt.plot(self.freqs, P, label=ch)
        plt.xlim(*xlim)
        plt.ylim(*ylim)
        plt.xlabel("Frequency (Hz)")
        plt.ylabel("Power spectral density (µV²/Hz)")
        plt.title(f"{condition} – Welch PSD")
        plt.legend()
        plt.show()

    def plot_all_epochs(self, channel: str = "TP9", ylim=(-100, 100)):
        n_cond = len(self.events)
        if n_cond == 0:
            print("⚠️ No conditions available to plot.")
            return
        fig, axes = plt.subplots(n_cond, 1, figsize=(12, 4 * n_cond), dpi=100, squeeze=False)
        fig.suptitle(f"EEG epochs – {channel}", fontsize=16)
        for i, (code, trials_by_channel) in enumerate(self.events.items()):
            trials = trials_by_channel.get(channel, [])
            ax = axes[i][0]
            for trial in trials:
                t = np.linspace(0, self.length, len(trial))
                ax.plot(t, trial, alpha=0.5)
            label = self.marker_dict.get(code, f"Code {code}")
            ax.set_title(f"{label} – {len(trials)} trials")
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Amplitude (μV)")
            ax.set_ylim(*ylim)
            ax.grid(True)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

    def plot_every_trial(self, channel: str = "TP9", ylim=(-100, 100)):
        total = 0
        for code, channel_trials in self.events.items():
            trials = channel_trials.get(channel, [])
            label  = self.marker_dict.get(code, f"Code {code}")
            for i, trial in enumerate(trials):
                t = np.linspace(0, self.length, len(trial))
                plt.figure(figsize=(10, 3), dpi=100)
                plt.plot(t, trial, label=f"{label} – Trial {i+1}")
                plt.title(f"{label} – Trial {i+1} ({channel})")
                plt.xlabel("Time (s)")
                plt.ylabel("Amplitude (μV)")
                plt.ylim(*ylim)
                plt.grid(True)
                plt.tight_layout()
                plt.show()
                total += 1
        print(f"✅ Plotted {total} trials for channel '{channel}'")

    def plot_every_trial_all_channels(self, ylim=(-100, 100)):
        total = 0
        for code, channel_trials in self.events.items():
            label = self.marker_dict.get(code, f"Code {code}")
            for channel, trials in channel_trials.items():
                for i, trial in enumerate(trials):
                    t = np.linspace(0, self.length, len(trial))
                    plt.figure(figsize=(10, 3), dpi=100)
                    plt.plot(t, trial, label=f"{label} – Trial {i+1} – {channel}")
                    plt.title(f"{label} – Trial {i+1} ({channel})")
                    plt.xlabel("Time (s)")
                    plt.ylabel("Amplitude (μV)")
                    plt.ylim(*ylim)
                    plt.grid(True)
                    plt.tight_layout()
                    plt.show()
                    total += 1
        print(f"✅ Plotted {total} trial-channel waveforms")

    def diagnose_trials(self):
        print("\n📋 Trial Diagnostics:")
        if hasattr(self, 'beh') and 'Stimulus' in self.beh.columns:
            beh_counts = self.beh["Stimulus"].value_counts()
            print("🧪 Behavioral stimulus counts:")
            for stim, count in beh_counts.items():
                print(f"  {stim}: {count}")
        else:
            print("⚠️ Behavioral data not available or missing 'Stimulus' column")
        print("\n🔗 EEG-aligned trials per condition:")
        for code in self.events:
            stim = self.marker_dict.get(code, f"Code {code}")
            trial_count = len(next(iter(self.events[code].values()), []))
            print(f"  {stim}: {trial_count} trials")
        print("\n🧹 Clean windows retained per condition:")
        for code in self.windows:
            stim = self.marker_dict.get(code, f"Code {code}")
            ch_counts = {ch: len(ws) for ch, ws in self.windows[code].items()}
            avg_clean = np.mean(list(ch_counts.values()))
            print(f"  {stim}: ~{int(avg_clean)} clean trials (avg per channel)")

    def tag_noise_levels(self, thresholds=(50, 150)):
        self.noise_labels = {}
        for code, channel_trials in self.windows.items():
            self.noise_labels[code] = {}
            for ch, trials in channel_trials.items():
                self.noise_labels[code][ch] = [
                    "low" if np.ptp(t) < thresholds[0]
                    else "medium" if np.ptp(t) < thresholds[1]
                    else "high"
                    for t in trials
                ]
        print("✅ Noise levels tagged. No exclusions applied.")

    def get_trials_by_noise(self, levels=("low", "medium"), condition=None, channel="TP9"):
        if not hasattr(self, "noise_labels"):
            print("⚠️ Run `tag_noise_levels()` first.")
            return []
        matched = []
        for code, ch_labels in self.noise_labels.items():
            stim = self.marker_dict.get(code, f"Code {code}")
            if condition and stim != condition:
                continue
            if channel not in ch_labels:
                continue
            for trial, lbl in zip(self.windows[code][channel], ch_labels[channel]):
                if lbl in levels:
                    matched.append(trial)
        print(f"✅ Retrieved {len(matched)} trials at noise levels {levels}" +
              (f" for '{condition}' @ {channel}" if condition else f" @ {channel}"))
        return matched

    def plot_trials(self, trials, title="EEG Trials", color="slateblue"):
        if not trials:
            print("⚠️ No trials to plot.")
            return
        plt.figure(figsize=(10, 5))
        for i, trial in enumerate(trials):
            plt.plot(trial, alpha=0.5, label=f"Trial {i+1}", color=color)
        plt.title(title)
        plt.xlabel("Time (samples)")
        plt.ylabel("Amplitude (μV)")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    def dataframe(self):
        return self.band_df.copy()


# ─────────────────────────────────────────────────────────────────────────────

class GrandAverageRecording:
    """
    Combines multiple Recording objects into a grand average across participants.

    Averages:
      - PSDs per condition/channel  (mean across participants)
      - Relative band power values  (mean across participants, stored in band_df)

    Usage:
        grand = GrandAverageRecording(sessions)
        grand.plot("circle")                  # grand-average PSD
        grand.dataframe()                      # grand-average band power table
        grand.plot_band_power("alpha")         # bar chart across conditions & channels
    """

    def __init__(self, recordings: list):
        if not recordings:
            raise ValueError("No Recording objects provided.")

        self.n = len(recordings)
        print(f"Building grand average from {self.n} participant(s)...")

        # Use the first valid recording as reference for metadata
        ref = recordings[0]
        self.freqs       = ref.freqs
        self.marker_dict = ref.marker_dict
        self.bands       = ref.bands

        # ── Average PSDs ──────────────────────────────────────────────────────
        # psd[condition_code][channel] = mean PSD array
        self.psd = {}
        all_conditions = set()
        for rec in recordings:
            all_conditions.update(rec.psd.keys())

        for code in all_conditions:
            label = self.marker_dict.get(code, f"Code {code}")
            self.psd[code] = {}
            # Collect all channels seen for this condition
            all_channels = set()
            for rec in recordings:
                if code in rec.psd:
                    all_channels.update(rec.psd[code].keys())

            for ch in all_channels:
                arrays = [
                    rec.psd[code][ch]
                    for rec in recordings
                    if code in rec.psd
                    and ch in rec.psd[code]
                    and isinstance(rec.psd[code][ch], np.ndarray)
                ]
                if arrays:
                    # Trim to shortest length in case of minor shape differences
                    min_len = min(a.shape[0] for a in arrays)
                    self.psd[code][ch] = np.mean([a[:min_len] for a in arrays], axis=0)
                    if self.freqs is not None and len(self.freqs) > min_len:
                        self.freqs = self.freqs[:min_len]
                else:
                    self.psd[code][ch] = None

            n_valid = sum(1 for rec in recordings if code in rec.psd)
            print(f"  '{label}': averaged PSDs from {n_valid} participant(s)")

        # ── Average band power DataFrames ─────────────────────────────────────
        band_dfs = [rec.band_df for rec in recordings if not rec.band_df.empty]
        if band_dfs:
            combined = pd.concat(band_dfs, ignore_index=True)
            band_cols = [c for c in combined.columns if c not in ('Condition', 'Channel')]
            self.band_df = (
                combined
                .groupby(['Condition', 'Channel'])[band_cols]
                .mean()
                .reset_index()
            )
            self.n_per_condition = (
                combined
                .groupby('Condition')['Channel']
                .count()
                .div(combined['Channel'].nunique())
                .to_dict()
            )
        else:
            self.band_df = pd.DataFrame()
            self.n_per_condition = {}
            print("⚠️ No valid band power data found across participants.")

        print("✅ Grand average ready.")

    def plot(self, condition: str, ylim=(0, 150), xlim=(0, 40)):
        """Grand-average Welch PSD for one condition, all channels."""
        inv  = {v: k for k, v in self.marker_dict.items()}
        code = inv.get(condition)
        if code is None or code not in self.psd:
            print(f"⚠️ Condition '{condition}' not found.")
            return
        n = self.n_per_condition.get(condition, self.n)
        plt.figure(dpi=120)
        for ch, P in self.psd[code].items():
            if P is None:
                continue
            plt.plot(self.freqs, P, label=ch)
        plt.xlim(*xlim)
        plt.ylim(*ylim)
        plt.xlabel("Frequency (Hz)")
        plt.ylabel("Power spectral density (µV²/Hz)")
        plt.title(f"{condition} – Grand Average Welch PSD  (n={int(n)})")
        plt.legend()
        plt.show()

    def dataframe(self):
        """Returns the grand-average relative band power table."""
        return self.band_df.copy()

    def plot_band_power(self, band: str = "alpha", ylim=None):
        """
        Bar chart of mean relative power for one frequency band,
        grouped by condition, split by channel.
        """
        if self.band_df.empty:
            print("⚠️ No band power data available.")
            return
        if band not in self.band_df.columns:
            print(f"⚠️ Band '{band}' not found. Available: {[c for c in self.band_df.columns if c not in ('Condition','Channel')]}")
            return

        pivot = self.band_df.pivot(index='Condition', columns='Channel', values=band)
        ax = pivot.plot(kind='bar', figsize=(10, 5), dpi=100)
        ax.set_title(f"Grand Average Relative {band.capitalize()} Power  (n={self.n})")
        ax.set_ylabel("Relative Power")
        ax.set_xlabel("Condition")
        if ylim:
            ax.set_ylim(*ylim)
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()

## Process All Participants

Reads every file pair, runs the full pipeline per participant, then builds the grand average.

In [ ]:
sessions = []

for i, (eeg_path, beh_path) in enumerate(participant_files, start=1):
    print(f"\n── Participant {i}/{N_PARTICIPANTS} ─────────────────────────")
    try:
        eeg_df  = load_eeg(eeg_path)
        beh_df  = load_behav(beh_path)

        # Build stimulus-to-code map from this participant's behavioural file
        stim_labels = beh_df["Stimulus"].unique()
        marker_map  = {label: i + 1 for i, label in enumerate(stim_labels)}

        rec = Recording(
            eeg_df,
            beh_df,
            marker_map,
            length=20,
            min_freq=1,
            max_freq=30
        )
        sessions.append(rec)
        print(f"   ✓ Done")
    except Exception as e:
        print(f"   ✗ Skipped — {e}")

print(f"\nSuccessfully processed {len(sessions)}/{N_PARTICIPANTS} participant(s).")

# Build grand average
grand = GrandAverageRecording(sessions)

## Grand Average Plots

In [ ]:
# Grand-average Welch PSD for every condition
for condition in grand.dataframe()["Condition"].unique():
    grand.plot(condition, ylim=(0, 250), xlim=(0, 40))

In [ ]:
# Grand-average relative band power — one chart per band
for band in ["delta", "theta", "alpha", "beta"]:
    grand.plot_band_power(band)

In [ ]:
# Grand-average band power table
grand.dataframe()

## Optional: Inspect a Single Participant

Use `sessions[i]` (0-indexed) to look at one participant before grand averaging.

In [ ]:
PARTICIPANT_INDEX = 0   # change to inspect a different participant

if PARTICIPANT_INDEX < len(sessions):
    s = sessions[PARTICIPANT_INDEX]
    s.diagnose_trials()
    display(s.dataframe())
    for condition in s.dataframe()["Condition"].unique():
        s.plot(condition, ylim=(0, 250), xlim=(0, 40))
else:
    print(f"Index {PARTICIPANT_INDEX} out of range — only {len(sessions)} session(s) loaded.")